# ACA 2 — Parte 2: Machine Learning (clasificación de causa BTI)

**Problema:** dado el texto y los datos operativos de un incidente, predecir su **categoría de causa (BTI)** — clasificación multiclase supervisada.

Por el desbalanceo (132 categorías, 58 con <10 ejemplos), se **limpian las etiquetas** y se modelan las categorías con soporte suficiente, agrupando la cola larga en **"otros"**. Ejecuta *Entorno de ejecución → Ejecutar todo*.

## 1. Conectar Drive y localizar la base

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, sqlite3, re, unicodedata
import pandas as pd, numpy as np
RUTA_DB='/content/drive/MyDrive/Especializacion IA/incidentes_anonimizado.db'
if not os.path.exists(RUTA_DB):
    validos=[c for c in glob.glob('/content/drive/MyDrive/**/incidentes_anonimizado.db', recursive=True) if os.path.isfile(c)]
    if validos: RUTA_DB=max(validos,key=os.path.getsize)
print('USANDO:',RUTA_DB,'|',round(os.path.getsize(RUTA_DB)/1e6,1),'MB')

## 2. Cargar datos y construir el target (limpieza + Top-N + 'otros')
Normaliza la etiqueta y agrupa en **'otros'** las categorías con menos de `MIN_EJEMPLOS` registros.

In [ ]:
con=sqlite3.connect(RUTA_DB)
ing=pd.read_sql('SELECT * FROM "ingresadas"', con); ing.columns=ing.columns.str.lower().str.strip()

def limpiar(x):
    x=str(x).strip().lower()
    x=''.join(c for c in unicodedata.normalize('NFKD',x) if not unicodedata.combining(c))  # quita acentos
    return re.sub(r'\s+',' ',x)

ing['target']=ing['bti_clasificacion'].apply(limpiar)
ing=ing[~ing['target'].isin(['','nan','sin_clasificar','sin clasificar'])]

MIN_EJEMPLOS=50           # categorias con menos de esto -> 'otros'
conteo=ing['target'].value_counts()
frecuentes=conteo[conteo>=MIN_EJEMPLOS].index
ing['target_modelo']=np.where(ing['target'].isin(frecuentes), ing['target'], 'otros')

print('Registros:',len(ing))
print('Clases tras agrupar (incluye "otros"):', ing['target_modelo'].nunique())
print('\nDistribucion del target del modelo:')
print(ing['target_modelo'].value_counts().to_string())

## 3. Features: texto (TF-IDF) + variables operativas
Combinamos el resumen del problema con prioridad/país/marca en un solo campo de texto.

In [ ]:
def campo(c): return ing[c].astype(str) if c in ing.columns else ''
ing['texto']=(campo('resumen_problema')+' '+campo('priorizacion')+' '+campo('pais')+' '+campo('marca')).str.lower()
X=ing['texto']; y=ing['target_modelo']
print('Ejemplo de texto:', X.iloc[0][:120])
print('Clases:', y.nunique(), '| Registros:', len(y))

## 4. Entrenar y evaluar (Regresión Logística con manejo de desbalanceo)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
vec=TfidfVectorizer(max_features=5000, ngram_range=(1,2))
Xtr_v=vec.fit_transform(Xtr); Xte_v=vec.transform(Xte)
clf=LogisticRegression(max_iter=2000, class_weight='balanced')
clf.fit(Xtr_v,ytr)
pred=clf.predict(Xte_v)
print(classification_report(yte,pred, zero_division=0))

## 5. Matriz de confusión (clases más frecuentes)

In [ ]:
import matplotlib.pyplot as plt
top=y.value_counts().head(10).index.tolist()
cm=confusion_matrix(yte,pred,labels=top)
fig,ax=plt.subplots(figsize=(8,7))
im=ax.imshow(cm,cmap='Blues')
ax.set_xticks(range(len(top))); ax.set_xticklabels(top,rotation=90,fontsize=8)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top,fontsize=8)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusion (top 10 clases)')
for i in range(len(top)):
    for j in range(len(top)): ax.text(j,i,cm[i,j],ha='center',va='center',fontsize=7)
plt.colorbar(im); plt.tight_layout(); plt.show()

## 6. Conclusiones (completar con los resultados)

- Desempeño global (accuracy / F1 macro): _completar con el classification_report_.
- Clases que el modelo predice bien vs. las que confunde (ver matriz): _completar_.
- Limitaciones: desbalanceo, etiquetas ruidosas y la clase 'otros' que agrupa la cola larga.
- Recomendaciones: estandarizar la categorización BTI en el origen y reforzar las clases con pocos ejemplos.

> Lo importante (como indicó el docente) no es un modelo perfecto, sino aplicar correctamente el proceso: formulación, preparación, entrenamiento, evaluación e interpretación.